# Session 13: カスタムコントローラ
# Session 13: Custom Controller

## 目標 / Objective

`send_rc_control()` + `get_telemetry()` で外部 PID ループを実装する。

Implement an external PID loop using `send_rc_control()` + `get_telemetry()`.

## この Notebook で学ぶこと / What You'll Learn

| トピック | 説明 |
|---------|------|
| 外部制御ループ | Python から RC 制御で操作 |
| リアルタイム PID | テレメトリフィードバックの PID |
| 位置保持 | 外部 PID で定位飛行 |

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

import sys
from pathlib import Path
_vpython_dir = Path().absolute().parent.parent.parent / 'simulator' / 'vpython' / 'control'
if str(_vpython_dir) not in sys.path:
    sys.path.insert(0, str(_vpython_dir))
from pid import PID

from stampfly_edu import connect_or_simulate

print("Ready! / 準備完了！")

## 1. 外部制御ループの概念 / External Control Loop Concept

```
PC (Python)                        StampFly (ESP32)
┌─────────────────────┐            ┌──────────────────┐
│                     │   WiFi     │                  │
│ [Custom PID] ───────│──RC ctrl──→│ [Inner loops]    │
│      ↑              │            │    ↓             │
│      └──────────────│←telemetry──│ [Sensors]        │
│                     │   400Hz    │                  │
└─────────────────────┘            └──────────────────┘
```

`send_rc_control()` は StampFly の内部コントローラ（姿勢/レート）の上に
外部から速度指令を送ります。

In [ ]:
# External position hold controller
# 外部位置保持コントローラ

drone = connect_or_simulate()
drone.takeoff()

# Target position
target_x = 0.0  # m
target_y = 0.0  # m

# Position PID controllers
pid_x = PID(Kp=30.0, Ti=3.0, Td=0.5, output_min=-50, output_max=50)
pid_y = PID(Kp=30.0, Ti=3.0, Td=0.5, output_min=-50, output_max=50)

# Control loop
dt = 0.05  # 20Hz control rate
duration = 10.0
n_steps = int(duration / dt)

log = {"time": [], "x": [], "y": [], "cmd_fb": [], "cmd_lr": []}

for i in range(n_steps):
    telem = drone.get_telemetry()
    x = telem.get("x", 0.0)
    y = telem.get("y", 0.0)
    
    # PID outputs (velocity commands)
    cmd_fb = int(pid_x.update(target_x, x, dt))   # forward/backward
    cmd_lr = int(pid_y.update(target_y, y, dt))    # left/right
    
    # Send RC control
    drone.send_rc_control(
        left_right_velocity=cmd_lr,
        forward_backward_velocity=cmd_fb,
        up_down_velocity=0,
        yaw_velocity=0,
    )
    
    log["time"].append(i * dt)
    log["x"].append(x)
    log["y"].append(y)
    log["cmd_fb"].append(cmd_fb)
    log["cmd_lr"].append(cmd_lr)
    
    time.sleep(dt)

# Stop and land
drone.send_rc_control(0, 0, 0, 0)
drone.land()

print(f"Control loop completed: {n_steps} iterations")

In [ ]:
# Plot results
# 結果をプロット
import pandas as pd
df = pd.DataFrame(log)

fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

axes[0].plot(df['time'], df['x'], 'b-', label='X')
axes[0].plot(df['time'], df['y'], 'r-', label='Y')
axes[0].axhline(y=target_x, color='b', linestyle='--', alpha=0.5)
axes[0].axhline(y=target_y, color='r', linestyle='--', alpha=0.5)
axes[0].set_ylabel('Position (m) / 位置')
axes[0].set_title('External Position Hold / 外部位置保持')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(df['time'], df['cmd_fb'], 'b-', label='FB command')
axes[1].plot(df['time'], df['cmd_lr'], 'r-', label='LR command')
axes[1].set_ylabel('RC command / RC 指令')
axes[1].set_xlabel('Time (s) / 時間')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. 考察課題 / Discussion Questions

1. **制御レート**: 20Hz と 50Hz でどちらの性能が良いか？なぜ？
2. **遅延の影響**: WiFi 通信による遅延は制御にどう影響するか？
3. **内部 vs 外部**: 外部コントローラが内部コントローラより有利な場面はあるか？

---

1. **Control Rate**: Which performs better, 20Hz or 50Hz? Why?
2. **Delay Impact**: How does WiFi communication delay affect control?
3. **Internal vs External**: When is external control advantageous over internal?

In [ ]:
drone.end()